# Multi-objective fit of transient photoluminescence (TrPL) and transient microwave conductivity (trMC) with rate equations

This example demonstrates how to fit transient photoluminescence (TrPL) and transient microwave conductivity (trMC) data simultaneously using a multi-objective optimization approach. The model is based on the rate equations for charge carrier dynamics in semiconductors, which include trapping and detraping processes.
The model is described by the following set of differential equations:

$$\frac{dn}{dt}  = G - k_{trap} n (Bulk_{tr} - n_t) - k_{direct} n (p + p_0)$$
$$\frac{dn_t}{dt}= k_{trap} n (Bulk_{tr} - n_t) - k_{detrap} n_t (p + p_0)$$
$$\frac{dp}{dt} = G - k_{detrap} n_t (p + p_0) - k_{direct} n (p + p_0)$$

where $n$ and $p$ are the electron and hole charge carrier densities, $G$ is the generation rate in m&#8315;&#179; s&#8315;&#185;, k<sub>trap</sub> and k<sub>detrap</sub> are the trapping and detraping rates in m&#179; s&#8315;&#185;, and k<sub>direct</sub> is the bimolecular/band-to-bad recombination rate in m&#179; s&#8315;&#185;.
The TrPL signal is given by:
$$ TrPL = I_{PL} k_{direct} n (p + p_0)$$
where $I_{PL}$ is a scaling factor for the PL signal.  
The TrMC signal is given by:
$$ TrMC = I_{MC} * (r_{\mu}*n + p) $$
where $r_{\mu}$ is the mobility ratio and $I_{MC}$ is a scaling factor for the TrMC signal.

The data fitted in this notebook is taken from the following publication:
[C. Kupfer et al., Journal of Material Chemistry C, 2024, 12, 95-102](https://doi.org/10.1039/D3TC03867J)

In [ ]:
# Import necessary libraries
import warnings, os, sys, shutil
# remove warnings from the output
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from numpy.random import default_rng
import torch, copy, uuid
import ax, logging
from ax.utils.notebook.plotting import init_notebook_plotting, render
init_notebook_plotting() # for Jupyter notebooks

try:
    from optimpv import *
    from optimpv.axBOtorch.axUtils import *
    from optimpv.RateEqfits.RateEqAgent import RateEqAgent
    from optimpv.RateEqfits.RateEqModel import *
    from optimpv.RateEqfits.Pumps import *
except Exception as e:
    sys.path.append('../') # add the path to the optimpv module
    from optimpv import *
    from optimpv.axBOtorch.axUtils import *
    from optimpv.RateEqfits.RateEqAgent import RateEqAgent
    from optimpv.RateEqfits.RateEqModel import *
    from optimpv.RateEqfits.Pumps import *


## Define the parameters for the simulation

In [ ]:
params = []

k_direct = FitParam(name = 'k_direct', value = 3.9e-17, bounds = [1e-17,1e-15], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$k_{\text{direct}}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log')
params.append(k_direct)

k_trap = FitParam(name = 'k_trap', value = 4e-18, bounds = [1e-19,1e-16], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$k_{\text{trap}}$', unit='m$^{3}$ s$^{-1}$', axis_type = 'log')
params.append(k_trap)

k_detrap = FitParam(name = 'k_detrap', value = 3.1e-18, bounds = [1e-19,1e-16], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$k_{\text{detrap}}$', unit='s$^{-1}$', axis_type = 'log')
params.append(k_detrap)

N_t_bulk = FitParam(name = 'N_t_bulk', value = 1.6e23, bounds = [1e22,5e23], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$N_{\text{t,bulk}}$', unit='m$^{-3}$', axis_type = 'log')
params.append(N_t_bulk)

I_factor_PL = FitParam(name = 'I_factor_PL', value = 1e-32, bounds = [1e-33,1e-31], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$I_{\text{PL}}$', unit='-', axis_type = 'log')
params.append(I_factor_PL)

I_factor_MC = FitParam(name = 'I_factor_MC', value = 2.2e-26, bounds = [1e-27,1e-25], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$I_{\text{PL}}$', unit='-', axis_type = 'log')
params.append(I_factor_MC)

ratio_mu = FitParam(name = 'ratio_mu', value = 4.2, bounds = [1,10], log_scale = True, rescale = True, value_type = 'float', type='range', display_name=r'$\mu_{\text{ratio}}$', unit='-', axis_type = 'linear')
params.append(ratio_mu)


## Load and prepare the experimental data

In [ ]:
# Define the path to the data 
curr_dir = os.getcwd()
parent_dir = os.path.abspath('../') # path to the parent directory if not in Notebooks use os.getcwd()
path2data  = os.path.join(parent_dir,'Data','perovskite_trPL_trMC')
filenames = ['S25D1_L532_F0.csv','S25D1_L532_F1.csv','S25D1_L532_F2.csv'] # list of filenames to be analyzed
res_dir = os.path.join(parent_dir,'temp') # path to the results directory

# Select Gfracs used for the data
Gfracs = [1, 0.552, 0.290, 0.136, 0.087]


In [ ]:
# Create a class that contains to do some basic data processing on the data
class Experiment:
    """ A set of measurements """
    def __init__(self, path2data, filenames, Gfracs, laserCenter=0, num_pts=1e3, take_log=False):
        self.path2data = path2data
        self.filenames = filenames
        self.Gfracs = Gfracs
        self.laserCenter = laserCenter
        self.num_pts = num_pts
        self.take_log = take_log
        
        self.get_data()
        pass
    
    
    def get_data(self):
        self.X_raw, self.y_raw_MW, self.y_raw_PL = [],[],[]
        for filename in self.filenames:
            # Create empty lists to store data
            X,y_MW, y_PL = [],[],[]
            
            #Load file and extract data
            with open(os.path.join(self.path2data, filename)) as f:
                for line in f:
                    tmp=line.strip("\n").split(",")
                    X.append(float(tmp[3]))
                    y_MW.append(float(tmp[4]))
                    
                    if len(tmp)>8:
                        y_PL.append(float(tmp[10]))
                    else:
                        raise ValueError("The file does not contain PL data")
            
            # Create output arrays
            self.X_raw.append(np.array(X))
            self.y_raw_MW.append(np.array(y_MW))
            self.y_raw_PL.append(np.array(y_PL))
        pass
    
    
    def process_data(self, cut_rise=False, cut_time=None, cut_sigma=False):
        # Create empty lists to store data
        X_out, y_out_MW, y_out_PL = [],[],[]
        self.X_processed, self.y_processed_MW, self.y_processed_PL = [],[],[]
        self.signalParams={}
        self.background_out_PL=[]
        # Data processing:
        for X, y_MW, y_PL, Gfrac in zip(self.X_raw, self.y_raw_MW, self.y_raw_PL, self.Gfracs):
                
            # Subtract the background from MW and PL data
            index = np.where(X<(-10e-9))  # Calculate the background from the average of the signal up to 10ns before the peak (this buffer is to prevent the rise of the peak to affect the background)
            self.signalParams["MW_background"] = np.mean(y_MW[index])
            self.signalParams["PL_background"] = np.mean(y_PL[index])
            self.signalParams["MW_sigma"] = np.std(y_MW[index])
            self.signalParams["PL_sigma"] = np.std(y_PL[index])
            
            
            y_MW = y_MW - self.signalParams["MW_background"]
            y_PL = y_PL - self.signalParams["PL_background"]
            print('PL Sigma {}, PL background {}, MW Sigma {}, MW background {}'.format(self.signalParams["PL_sigma"],self.signalParams["PL_background"],self.signalParams["MW_sigma"],self.signalParams["MW_background"]))
            
            # Find the peak position
            self.signalParams["index_max_MW"] = np.argmax(abs(y_MW))
            self.signalParams["index_max_PL"] = np.argmax(abs(y_PL))
            
            # Find the sign of the peak
            self.signalParams["sign_max_MW"] = np.sign(y_MW[self.signalParams["index_max_MW"]])
            self.signalParams["sign_max_PL"] = np.sign(y_PL[self.signalParams["index_max_PL"]])
            
            # Remove datapoints at the beginning of the signal
            if cut_rise == "MW":
                index = np.where(X >= X[self.signalParams["index_max_MW"]])
            elif cut_rise == "PL":
                index = np.where(X >= X[self.signalParams["index_max_PL"]])
            elif cut_rise == "Time":
                index = np.where(X > cut_time)
            else:
                index = np.where(X > self.laserCenter)
                
            X = X[index]
            # Remove datapoints before the laser peak from the MW and PL signal and make sure, that the peak is positive
            y_MW = y_MW[index]*self.signalParams["sign_max_MW"]
            y_PL = y_PL[index]*self.signalParams["sign_max_PL"]
            
            # Remove datapoints that aren't significant enough (in either measurement)
            if cut_sigma:
                sigma = float(cut_sigma)
                index = np.where((np.abs(y_MW)>sigma*self.signalParams["MW_sigma"]) & (np.abs(y_PL)>sigma*self.signalParams["PL_sigma"]))

                X = X[index]
                y_MW = y_MW[index]
                y_PL = y_PL[index]
            
            
            # Interpolate to get num_pts
            X_interp = np.geomspace(X[1],X[-1],int(self.num_pts))

            # Add 0 to the beginning of X_interp
            X_interp = np.insert(X_interp,0,0)
            y_interp_MW = np.interp(X_interp,X,y_MW)
            y_interp_PL = np.interp(X_interp,X,y_PL)

            # Take the log of the data
            if self.take_log:
                y_interp_MW = np.log10(y_interp_MW)
                y_interp_PL = np.log10(y_interp_PL)
                
                # Remove all data points where either signal is NaN
                mask_NaNs = np.logical_or(np.isnan(y_interp_PL), np.isnan(y_interp_MW))
                X_interp = X_interp[~mask_NaNs]
                y_interp_MW = y_interp_MW[~mask_NaNs]
                y_interp_PL = y_interp_PL[~mask_NaNs]
                print('Removed {} Data Points while taking the logarithm!'.format(np.count_nonzero(mask_NaNs)))
            
            # Append the data to the output
            for i in range(len(X_interp)):
                X_out.append([X_interp[i],Gfrac])
                y_out_MW.append(y_interp_MW[i])
                y_out_PL.append(y_interp_PL[i])
                self.background_out_PL.append(self.signalParams["PL_sigma"]*np.sqrt(2/np.pi))
                
            self.X_processed.append(np.array(X_interp))
            self.y_processed_MW.append(np.array(y_interp_MW))
            self.y_processed_PL.append(np.array(y_interp_PL))

        # Convert the output to arrays
        self.X = np.array(X_out)
        self.y_MW = np.array(y_out_MW)
        self.y_PL = np.array(y_out_PL)
        pass


In [ ]:
# Load the data and process it
data_exp = Experiment(path2data, filenames, Gfracs, laserCenter=2.8E-8, take_log=False)
data_exp.process_data(cut_rise=False, cut_time=None ,cut_sigma=0)
X = data_exp.X
y_MW = data_exp.y_MW
y_PL = data_exp.y_PL
back_PL = data_exp.background_out_PL
print(back_PL)
# Assign weights based on the signal strength
weight_PL = None#1/(np.abs(y_PL))
weight_MW = None #1/(np.abs(y_MW))


In [ ]:
# RateEqModel parameters
fpu = 1e3 # Frequency of the pump laser in Hz
N0 = 1.041e24 # Initial carrier density in m-3
background = 0 # Background illumination 
Gfracs = [1, 0.552, 0.290] # Gfracs used for the data

# Define the Agent and the target metric/loss function
metric = 'nrmse'
loss = 'linear'
pump_args = {'N0': N0, 'fpu': fpu , 'background' : background, }

RateEq = RateEqAgent(params, [X,X], [y_PL,y_MW], model = BTD_model, pump_model = initial_carrier_density, pump_args = pump_args, fixed_model_args = {}, metric = [metric,metric], loss = [loss,loss], threshold=[0.25,0.1],minimize=[True,True],exp_format=['trPL','trMC'],detection_limit=1e-5, weight=[weight_PL,weight_MW], compare_type ='log')
X1 = X[X[:,1]==Gfracs[0],:][:,0]
X2 = X[X[:,1]==Gfracs[1],:][:,0]
X3 = X[X[:,1]==Gfracs[2],:][:,0]
y_PL1 = y_PL[X[:,1]==Gfracs[0]]
y_PL2 = y_PL[X[:,1]==Gfracs[1]]
y_PL3 = y_PL[X[:,1]==Gfracs[2]]
y_MW1 = y_MW[X[:,1]==Gfracs[0]]
y_MW2 = y_MW[X[:,1]==Gfracs[1]]
y_MW3 = y_MW[X[:,1]==Gfracs[2]]
pump_args = {'N0': N0/Gfracs[0], 'fpu': fpu , 'background' : background, }
RateEq1 = RateEqAgent(params, [X1,X1], [y_PL1,y_MW1], model = BTD_model, pump_model = initial_carrier_density, pump_args = pump_args, fixed_model_args = {}, metric = [metric,metric], loss = [loss,loss], threshold=[0.25,0.1],minimize=[True,True],exp_format=['trPL','trMC'],detection_limit=1e-5, weight=None, compare_type ='log',name='Gfrac_'+str(Gfracs[0]))
pump_args = {'N0': N0/Gfracs[1], 'fpu': fpu , 'background' : background, }
RateEq2 = RateEqAgent(params, [X2,X2], [y_PL2,y_MW2], model = BTD_model, pump_model = initial_carrier_density, pump_args = pump_args, fixed_model_args = {}, metric = [metric,metric], loss = [loss,loss], threshold=[0.25,0.1],minimize=[True,True],exp_format=['trPL','trMC'],detection_limit=1e-5, weight=None, compare_type ='log',name='Gfrac_'+str(Gfracs[1]))
pump_args = {'N0': N0/Gfracs[2], 'fpu': fpu , 'background' : background, }
RateEq3 = RateEqAgent(params, [X3,X3], [y_PL3,y_MW3], model = BTD_model, pump_model = initial_carrier_density, pump_args = pump_args, fixed_model_args = {}, metric = [metric,metric], loss = [loss,loss], threshold=[0.25,0.1],minimize=[True,True],exp_format=['trPL','trMC'],detection_limit=1e-5, weight=None, compare_type ='log',name='Gfrac_'+str(Gfracs[2]))

# Not necessary, but here we run the simulation with the parameters to test the model
# Run the simulation with the parameters
y_PL_fit = RateEq.run(parameters={},exp_format='trPL')
y_MC_fit = RateEq.run(parameters={},exp_format='trMC')

# Make a plot with 2 subplots for PL and MW
fig, (ax1, ax2) = plt.subplots(1, 2,figsize=(10,5))

for i in range(len(data_exp.X_processed)):
    ax1.semilogx(data_exp.X_processed[i], data_exp.y_processed_PL[i], 'o', label=data_exp.Gfracs[i],color='C'+str(i),alpha=0.025)
    ax1.plot(X[X[:,1]==Gfracs[i],0], y_PL_fit[X[:,1]==Gfracs[i]],'-',label=str(Gfracs[i]), color = 'C'+str(i), linewidth=2)
    ax2.semilogx(data_exp.X_processed[i], data_exp.y_processed_MW[i], 'o', label=data_exp.Gfracs[i],color='C'+str(i),alpha=0.025)
    ax2.plot(X[X[:,1]==Gfracs[i],0], y_MC_fit[X[:,1]==Gfracs[i]],'-',label=str(Gfracs[i]), color = 'C'+str(i), linewidth=2)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('PL (a.u.)')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('MW (a.u.)')

In [ ]:
def multi_exp_approximation(x, y, A1, A2, A3, A4, tau1, tau2, tau3, tau4, beta2, beta3, beta4, y0):
    
    decay = A1*np.exp(-x/tau1) + A2*np.exp(-(x/tau2)**beta2) + A3*np.exp(-(x/tau3)**beta3) + A4*np.exp(-(x/tau4)**beta4) + y0
    return np.log10(decay)

# def multi_exp_approximation(x, y, A1, A2, A3, A4, tau1, tau2, tau3, tau4, beta2, beta3, beta4, y0):
    
#     decay = 10**A1*np.exp(-x/tau1) + 10**A2*np.exp(-(x/tau2)**beta2) + 10**A3*np.exp(-(x/tau3)**beta3) + 10**A4*np.exp(-(x/tau4)**beta4) + 10**(-y0)
#     return np.log10(decay)

In [ ]:
X_filter, y_PL_filter, y_MC_filter = [],[],[]
for i in range(len(Gfracs)):
    X_dum = X[X[:,1]==Gfracs[i],0]
    y_PL_dum = y_PL[X[:,1]==Gfracs[i]]
    y_MC_dum = y_MW[X[:,1]==Gfracs[i]]

    # select 


In [ ]:
# import curve fitting
print(y_PL)
from scipy.optimize import curve_fit
X_filter, y_PL_filter, y_MC_filter = [],[],[]
num_pts = 50
for i in range(len(Gfracs)):
    print(i)
    X_ = X[X[:,1]==Gfracs[i],:][:,0]
    y_ = y_PL[X[:,1]==Gfracs[i]]
    X_ = X_[y_>0]
    y_ = y_[y_>0]
    time = np.geomspace(np.min(X_[X_>0]),np.max(X_),num_pts-1)
    time = np.insert(time,0,0)
    print(y_)
    popt,_ = curve_fit(multi_exp_approximation, X_, np.log10(y_), maxfev=100000)
    
    # yMC_ =

    if i == 0:
        y_PL_filter = multi_exp_approximation(time, *popt)
        # time and Gfrac in X_filter
        X_filter = np.array([[time[j],Gfracs[i]] for j in range(len(time))])
    else:
        y_PL_filter = np.concatenate((y_PL_filter,multi_exp_approximation(time, *popt)))
        # time and Gfrac in X_filter
        X_filter = np.concatenate((X_filter,np.array([[time[j],Gfracs[i]] for j in range(len(time))])))
    

print(y_PL_filter)

In [ ]:
# import curve fitting
from scipy.optimize import curve_fit


# t_ = X1[y_PL1 > 0]
# y_PL1 = y_PL1[y_PL1 > 0]
# popt, _ = curve_fit(multi_exp_approximation, t_, np.log10(y_PL1), maxfev=100000)
t_ = X1[y_PL1 > 0]
y_PL1 = y_PL1[y_PL1 > 0]
popt, _ = curve_fit(multi_exp_approximation, t_, np.log10(y_PL1), maxfev=100000)
# t_ = X1[y_MW1 > 0]
# y_MW1 = y_MW1[y_MW1 > 0]
# popt, _ = curve_fit(multi_exp_approximation, t_, np.log10(y_MW1), maxfev=100000)
time = np.geomspace(np.min(X1[X1>0]),np.max(X1),50)
# add 0 to the beginning of time
time = np.insert(time,0,0)
spline_fit = multi_exp_approximation(time, *popt)

plt.figure()
plt.semilogx(t_, y_PL1, 'o', label='Data',color='C0')
# plt.semilogx(X1, y_MW1, 'o', label='Data',color='C0')
plt.plot(time, 10**spline_fit, 'o', label='Fit', color='C1', linewidth=2)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Time (s)')
plt.ylabel('PL (a.u.)')
plt.legend()
plt.show()
print(popt)

In [ ]:
print(RateEq.run_Ax(parameters={}))

## Run the optimization

In [ ]:
from optimpv.axBOtorch.axBOtorchOptimizer import axBOtorchOptimizer
from botorch.acquisition.multi_objective.logei import qLogExpectedHypervolumeImprovement     
from ax.modelbridge.transforms.standardize_y import StandardizeY
from ax.modelbridge.transforms.unit_x import UnitX
from ax.modelbridge.transforms.remove_fixed import RemoveFixed
from ax.modelbridge.transforms.log import Log
# from optimpv.axBOtorch.EGBO import EGBOAcquisition

model_gen_kwargs_list = None
parameter_constraints = None

model_kwargs_list = [{},{'torch_device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),'torch_dtype': torch.double,'botorch_acqf_class':qLogExpectedHypervolumeImprovement,'transforms':[RemoveFixed, Log,UnitX, StandardizeY]}]#,'acquisition_class':EGBOAcquisition}] # list of model kwargs for each model

optimizer = axBOtorchOptimizer(params = params, agents = [RateEq1,RateEq2,RateEq3], models = ['SOBOL','BOTORCH_MODULAR'],n_batches = [1,30], batch_size = [20,2], ax_client = None,  max_parallelism = -1, model_kwargs_list = model_kwargs_list, model_gen_kwargs_list = None, name = 'ax_opti')

In [ ]:
optimizer.optimize(True) # run the optimization with ax

In [ ]:
# get the best parameters and update the params list in the optimizer and the agent
ax_client = optimizer.ax_client # get the ax client
optimizer.update_params_with_best_balance() # update the params list in the optimizer with the best parameters
RateEq.params = optimizer.params # update the params list in the agent with the best parameters

# print the best parameters
print('Best parameters:')
for p in optimizer.params:
    print(p.name, 'fitted value:', p.value)
    

In [ ]:
# Plot optimization results
data = ax_client.experiment.fetch_data()
trace = ax_client.get_trace()
plt.figure()
plt.plot(trace)
# plt.plot(np.minimum.accumulate(data.df["mean"]), label="Best value seen so far")
plt.yscale("log")
plt.xlabel("Number of batches")
plt.ylabel("Hypervolume")
plt.title("Best value seen so far")

print("Best value seen so far is ", np.max(trace), "in batch ", np.argmax(ax_client.get_trace()))

In [ ]:
import matplotlib
# import itertools
from itertools import combinations
comb = list(combinations(optimizer.all_metrics, 2))
threshold_list = []
for i in range(len(optimizer.agents)):
    for j in range(len(optimizer.agents[i].threshold)):
        threshold_list.append(optimizer.agents[i].threshold[j])
threshold_comb = list(combinations(threshold_list, 2))
pareto = ax_client.get_pareto_optimal_parameters(use_model_predictions=False)
cm = matplotlib.colormaps.get_cmap('viridis')
df = get_df_from_ax(params, optimizer)
# create pareto df
dum_dic = {}
for eto in pareto.keys():
    for metr in optimizer.all_metrics:
        if metr not in dum_dic.keys():
            dum_dic[metr] = []
        dum_dic[metr].append(pareto[eto][1][0][metr])
df_pareto = pd.DataFrame(dum_dic)

for c,t_c in zip(comb,threshold_comb):
    plt.figure(figsize=(10, 10))
    plt.scatter(df[c[0]],df[c[1]],c=df.index, cmap=cm, marker='o', s=100) # plot the points with color according to the iteration
    cbar = plt.colorbar()
    cbar.set_label('Iteration')
    sorted_df = df_pareto.sort_values(by=c[0])
    plt.plot(sorted_df[c[0]],sorted_df[c[1]],'r') 
    plt.scatter(t_c[0],t_c[1],c='r', marker='x', s=100) # plot the threshold
    plt.xlabel(c[0])
    plt.ylabel(c[1])
    plt.xscale('log')
    plt.yscale('log')
    
    
    plt.show()


In [ ]:
# rerun the simulation with the best parameters
# Not necessary, but here we run the simulation with the parameters to test the model
# Run the simulation with the parameters
y_PL_fit = RateEq.run(parameters={},exp_format='trPL')
y_MC_fit = RateEq.run(parameters={},exp_format='trMC')

# Make a plot with 2 subplots for PL and MW
fig, (ax1, ax2) = plt.subplots(1, 2,figsize=(10,5))

for i in range(len(Gfracs)):
    ax1.semilogx(X[X[:,1]==Gfracs[i],0], y_PL[X[:,1]==Gfracs[i]]/np.max(y_PL), 'o', color='C'+str(i),alpha=0.025)
    ax1.plot(X[X[:,1]==Gfracs[i],0], y_PL_fit[X[:,1]==Gfracs[i]]/np.max(y_PL_fit),'-', color = 'C'+str(i), linewidth=2)
    ax2.semilogx(X[X[:,1]==Gfracs[i],0], y_MW[X[:,1]==Gfracs[i]]/np.max(y_MW), 'o', color='C'+str(i),alpha=0.025)
    ax2.plot(X[X[:,1]==Gfracs[i],0], y_MC_fit[X[:,1]==Gfracs[i]]/np.max(y_MC_fit),'-', color = 'C'+str(i), linewidth=2)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('PL (a.u.)')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('MW (a.u.)')
plt.show()


In [ ]:
# rerun the simulation with the best parameters
# Not necessary, but here we run the simulation with the parameters to test the model
# Run the simulation with the parameters
y_PL_fit = RateEq.run(parameters={},exp_format='trPL')
y_MC_fit = RateEq.run(parameters={},exp_format='trMC')

# Make a plot with 2 subplots for PL and MW
fig, (ax1, ax2) = plt.subplots(1, 2,figsize=(10,5))

for i in range(len(Gfracs)):
    ax1.semilogx(X[X[:,1]==Gfracs[i],0], y_PL[X[:,1]==Gfracs[i]], 'o', color='C'+str(i),alpha=0.025)
    ax1.plot(X[X[:,1]==Gfracs[i],0], y_PL_fit[X[:,1]==Gfracs[i]],'-', color = 'C'+str(i), linewidth=2)
    ax2.semilogx(X[X[:,1]==Gfracs[i],0], y_MW[X[:,1]==Gfracs[i]], 'o', color='C'+str(i),alpha=0.025)
    ax2.plot(X[X[:,1]==Gfracs[i],0], y_MC_fit[X[:,1]==Gfracs[i]],'-', color = 'C'+str(i), linewidth=2)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('PL (a.u.)')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('MW (a.u.)')
plt.show()

In [ ]:
print(RateEq.run_Ax(parameters={}))

In [ ]:
# Plot the density of the exploration of the parameters
# this gives a nice visualization of where the optimizer focused its exploration and may show some correlation between the parameters
plot_dens = True
if plot_dens:
    from optimpv.posterior.posterior import *
    best_parameters = {}
    for p in optimizer.params:
        best_parameters[p.name] = p.value

    fig_dens, ax_dens = plot_density_exploration(params, optimizer = optimizer, best_parameters = best_parameters, optimizer_type = 'ax')